<a href="https://colab.research.google.com/github/AndresCMontejo/Veterinaria_DOGtor_Agente_IA/blob/main/Dokky_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**¿COMO SE CONSTRUYÓ EL AGENTE?**

##Pasos
1. Conectar Drive
2. Encontrar los Docx
3. Extraer el texto
4. Añadir metadatos
5. Separar el system prompt
6. Dividir el texto en chunks
7. Generar embeddings
8. Guardarlos en Chroma
9. Hacer búsquedas de prueba
10. Conectar Gemini para redactar respuestas
11. Crear la interfaz

**1. Conectar Drive**

In [5]:
#Conectando con google drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**2. Encontrar los GDOC**

In [6]:
#Definiendo la ruta de los documentos
from pathlib import Path
DOCUMENTS_DIR = Path(
    "/content/drive/MyDrive/documentos_agente_de_ia/documentos"
)

if not DOCUMENTS_DIR.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta: {DOCUMENTS_DIR}"
    )

print("Carpeta encontrada:", DOCUMENTS_DIR)

Carpeta encontrada: /content/drive/MyDrive/documentos_agente_de_ia/documentos


In [13]:
#COMPROBANDO SI HAY ARCHIVOS EXISTENTES EN DOCUMENTOS
document_files = list(DOCUMENTS_DIR.glob("*.docx"))

print(f"Documentos encontrados: {len(document_files)}")

for file in document_files:
    print("-", file.name)

Documentos encontrados: 6
- servicios_veterinaria_dogtor.docx
- guia_convenios.docx
- politica_cancelaciones_reprogramacion.docx
- politica_privacidad_datos_paciente.docx
- instrucciones_pre_postconsulta.docx
- citas_agendamientos.docx


In [11]:
#Preparación de bibliotecas
!pip install -qU \
    python-docx \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-chroma \
    chromadb \
    google-genai

In [12]:
#Comprobando la llave API Key de gemini
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI")
if not GEMINI_API_KEY:
    raise ValueError(
        "No se encontró GEMINI_API_KEY en los secretos de Colab."
    )

print("Clave de Gemini cargada correctamente.")

Clave de Gemini cargada correctamente.


In [14]:
#COMPROBANDO EL FORMATO DE LOS ARCHIVOS
from pathlib import Path
BASE_DIR = Path(
    "/content/drive/MyDrive/documentos_agente_de_ia"
)
DOCUMENTS_DIR = BASE_DIR / "documentos"
PROMPT_DIR = BASE_DIR / "prompt"
print("Archivos de conocimiento:")
for archivo in DOCUMENTS_DIR.iterdir():
    print(
        f"- {archivo.name} | "
        f"Extensión: {archivo.suffix} | "
        f"Tamaño: {archivo.stat().st_size} bytes"
    )

print("\nArchivos de configuración:")
for archivo in PROMPT_DIR.iterdir():
    print(
        f"- {archivo.name} | "
        f"Extensión: {archivo.suffix} | "
        f"Tamaño: {archivo.stat().st_size} bytes"
    )

Archivos de conocimiento:
- DOCUMENTOS GDOC | Extensión:  | Tamaño: 4096 bytes
- servicios_veterinaria_dogtor.docx | Extensión: .docx | Tamaño: 9157 bytes
- guia_convenios.docx | Extensión: .docx | Tamaño: 12184 bytes
- politica_cancelaciones_reprogramacion.docx | Extensión: .docx | Tamaño: 321292 bytes
- politica_privacidad_datos_paciente.docx | Extensión: .docx | Tamaño: 11043 bytes
- instrucciones_pre_postconsulta.docx | Extensión: .docx | Tamaño: 12948 bytes
- citas_agendamientos.docx | Extensión: .docx | Tamaño: 11079 bytes

Archivos de configuración:
- Prompt GDOC | Extensión:  | Tamaño: 4096 bytes
- system_prompt.docx | Extensión: .docx | Tamaño: 9132 bytes


In [15]:
#COMPROBANDO LAS DEPENDENCIAS INSTALADAS
import docx
import chromadb
import langchain
import google.genai

print("python-docx:", docx.__version__)
print("chromadb:", chromadb.__version__)
print("langchain:", langchain.__version__)
print("Las bibliotecas principales están disponibles.")

python-docx: 1.2.0
chromadb: 1.5.9
langchain: 1.3.14
Las bibliotecas principales están disponibles.


**3. Extraer el texto**

In [25]:
#El siguiente codigo se utiliza para extraer el contenido de los archivos .docx, y las tablas, ya que cualquier error, el agente puede no funcionar como se espera.
from pathlib import Path
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table

def extraer_texto_docx(ruta_archivo: Path) -> str:
    """
    Extrae párrafos y tablas de un archivo DOCX
    conservando el orden original del documento.
    """
    documento = Document(ruta_archivo)
    partes = []

    for elemento in documento.iter_inner_content():

        # Extraer párrafos
        if isinstance(elemento, Paragraph):
            texto = elemento.text.strip()

            if texto:
                partes.append(texto)

        # Extraer tablas
        elif isinstance(elemento, Table):
            partes.append("[INICIO DE TABLA]")

            for fila in elemento.rows:
                celdas = []

                for celda in fila.cells:
                    texto_celda = " ".join(
                        parrafo.text.strip()
                        for parrafo in celda.paragraphs
                        if parrafo.text.strip()
                    )

                    celdas.append(texto_celda)

                if any(celdas):
                    partes.append(" | ".join(celdas))

            partes.append("[FIN DE TABLA]")

    return "\n".join(partes)

**4. Añadir metadatos**

In [26]:
#CREANDO METADATOS DE LOS DOCUMENTOS
METADATOS_DOCUMENTOS = {
    "servicios_veterinaria_dogtor.docx": {
        "Categoria": "Servicios",
        "Departamento responsable": "Dirección Médica",
        "Nivel de acceso": "Público",
        "Estado": "Vigente",
        "Versión": "1.0"
    },

    "politica_privacidad_datos_paciente.docx": {
        "Categoria": "Legal y Compliance",
        "Departamento responsable": "Departamento Jurídico",
        "Tipo de documento": "Política Corporativa",
        "Nivel de acceso": "Todos los colaboradores",
        "Estado": "Vigente",
        "Version": "1.0"
    },

    "politica_cancelaciones_reprogramacion.docx": {
        "Categoria": "Atención al Cliente",
        "Departamento responsable": "Recepción y Atención al Cliente",
        "Tipo de documento": "Política Corporativa",
        "Nivel de acceso": "Todos los colaboradores",
        "Estado": "Vigente",
        "Version": "1.0"
    },

    "instrucciones_pre_postconsulta.docx": {
        "Categoria": "Atención médica",
        "Departamento responsable": "Dirección médica",
        "Tipo de documento": "Política Corporativa",
        "Nivel de acceso": "Manual clínico",
        "Estado": "Vigente",
        "Version":"1.0"
    },

    "guia_convenios.docx": {
        "Categoria": "Convenios y Alianzas",
        "Departamento responsable": "Relaciones institucionales",
        "Tipo de documento": "Guia corporativa",
        "Nivel de acceso": "Todo los colaboradores",
        "Estado": "Vigente",
        "Version": "1.0"
    },

    "citas_agendamientos.docx": {
        "Categoria": "Atención al cliente",
        "Departamento responsable": "Recepción y Atención al Cliente",
        "Tipo de documento": "Preguntas Frecuentes",
        "Nivel de acceso": "Todos los colaboradores",
        "Estado": "Vigente",
        "Versión":"1.0"
    }
}

In [27]:
#CARGANDO LOS DOCUMENTOS
documentos_cargados = []

archivos_docx = sorted(DOCUMENTS_DIR.glob("*.docx"))

for ruta_archivo in archivos_docx:
    texto = extraer_texto_docx(ruta_archivo)

    metadatos = METADATOS_DOCUMENTOS.get(
        ruta_archivo.name,
        {
            "Categoria": "Sin clasificar",
            "Departamento responsable": "Sin especificar",
            "Tipo de documento": "Documento",
            "Nivel de acceso": "Público",
            "Estado": "Vigente",
            "Version": "1.0"
        }
    )

    metadatos = {
        **metadatos,
        "nombre_archivo": ruta_archivo.name,
        "ruta_origen": str(ruta_archivo),
        "formato": "docx"
    }

    documentos_cargados.append(
        {
            "texto": texto,
            "metadata": metadatos
        }
    )

print(f"Documentos cargados: {len(documentos_cargados)}")

Documentos cargados: 6


In [28]:
#PRUEBA, PARA LOS DOCUMENTOS QUE TENGAN UNICAMENTE TABLAS
for documento in documentos_cargados:
    texto = documento["texto"]
    archivo = documento["metadata"]["nombre_archivo"]

    cantidad_tablas = texto.count("[INICIO DE TABLA]")

    print(
        f"{archivo}: "
        f"{cantidad_tablas} tabla(s) detectada(s)"
    )

citas_agendamientos.docx: 0 tabla(s) detectada(s)
guia_convenios.docx: 1 tabla(s) detectada(s)
instrucciones_pre_postconsulta.docx: 2 tabla(s) detectada(s)
politica_cancelaciones_reprogramacion.docx: 1 tabla(s) detectada(s)
politica_privacidad_datos_paciente.docx: 0 tabla(s) detectada(s)
servicios_veterinaria_dogtor.docx: 0 tabla(s) detectada(s)


In [29]:
#MOSTRANDO LOS DOCUMENTOS QUE TENGAN UNICAMENTE TABLAS
for documento in documentos_cargados:
    texto = documento["texto"]
    archivo = documento["metadata"]["nombre_archivo"]

    if "[INICIO DE TABLA]" in texto:
        posicion = texto.find("[INICIO DE TABLA]")

        print("=" * 80)
        print("Archivo:", archivo)
        print("\nFragmento con tabla:\n")

        inicio = max(0, posicion - 200)
        fin = min(len(texto), posicion + 1200)

        print(texto[inicio:fin])
        print()

Archivo: guia_convenios.docx

Fragmento con tabla:

ROS DE GASTOS MÉDICOS)
Aceptamos y facturamos directamente a las siguientes aseguradoras. Si tu póliza está activa, no pagas en caja (solo el deducible o coaseguro que marque tu contrato, si aplica).
[INICIO DE TABLA]
Aseguradora | Tipo de cobertura | Beneficio DOGtor | Proceso de cobro
GNP Seguros (Póliza Mascotas) | Accidentes, cirugías, hospitalización y estudios de diagnóstico. | Convenio preferente: Sin tope de gastos notariales para la reclamación. | Cobramos directamente a GNP. Tú solo firmas la factura y pagas el deducible establecido.
Mapfre México (Protección Animal) | Consultas de especialidad, medicamentos recetados y urgencias. | 10% de descuento en el deducible si pagas con tarjeta DOGtor (afiliación gratuita). | Emitimos factura CFDI a nombre de Mapfre y te damos el comprobante para tu ajuste.
AXA Seguros (Salud para Mascotas) | Cobertura integral (incluye consultas generales y vacunación anual). | Cobertura ampliada: I

In [30]:
#Comprobando cada documento
for numero, documento in enumerate(documentos_cargados, start=1):
    texto = documento["texto"]
    metadata = documento["metadata"]

    print("=" * 80)
    print(f"DOCUMENTO {numero}")
    print("Archivo:", metadata["nombre_archivo"])
    print("Categoría:", metadata["Categoria"])
    print("Departamento:", metadata["Departamento responsable"])
    print("Caracteres extraídos:", len(texto))
    print("\nVista previa:")
    print(texto[:7000])
    print()

DOCUMENTO 1
Archivo: citas_agendamientos.docx
Categoría: Atención al cliente
Departamento: Recepción y Atención al Cliente
Caracteres extraídos: 6438

Vista previa:
PREGUNTAS FRECUENTES (FAQ) – CITAS Y AGENDAMIENTOS
VERSIÓN: 1.0
FECHA DE VIGENCIA: 8 de Octubre de 2026
EMPRESA: Veterinaria DOGtor S.A. de C.V.
DIRECCIÓN SEDE CENTRAL: Av. Paseo de la Reforma 1234, Colonia Juárez, Alcaldía Cuauhtémoc, CDMX.
Categoría: Atención al cliente
Departamento responsable: Recepción y Atención al Cliente
Tipo de documento: Preguntas Frecuentes
Nivel de acceso: Todos los colaboradores
Estado: Vigente
Palabras clave:
Citas, horarios, costos, agendar, urgencias
INTRODUCCIÓN
En Veterinaria DOGtor sabemos que la salud de tu peludo es prioridad. Por eso, hemos diseñado un sistema de agendamiento ágil, transparente y adaptado a tu ritmo de vida en la Ciudad de México. A continuación, resolvemos las dudas más comunes para que agendar, modificar o prepararte para tu cita sea pan comido (o mejor dicho, croque

**5. Separar el system prompt**

In [31]:
#CARGANDO EL SYSTEM PROMPT
archivos_prompt = list(PROMPT_DIR.glob("*.docx"))

if len(archivos_prompt) != 1:
    raise ValueError(
        f"Se esperaba un único archivo DOCX en la carpeta prompt, "
        f"pero se encontraron {len(archivos_prompt)}."
    )

ruta_system_prompt = archivos_prompt[0]
SYSTEM_PROMPT = extraer_texto_docx(ruta_system_prompt)

print("System prompt cargado correctamente.")
print("Archivo:", ruta_system_prompt.name)
print("Caracteres:", len(SYSTEM_PROMPT))
print("\nVista previa:")
print(SYSTEM_PROMPT[:700])

System prompt cargado correctamente.
Archivo: system_prompt.docx
Caracteres: 3952

Vista previa:
PROMPT DE SISTEMA PARA EL AGENTE VETERINARIO DOGtor
Versión: 1.0 | Fecha: 26/07/2026
1. Rol y Personalidad:
Eres "Dokky", el asistente virtual oficial de Veterinaria DOGtor S.A. de C.V., ubicada en Av. Paseo de la Reforma 1234, CDMX. Tu misión es brindar información clara, cálida y precisa a los tutores (dueños) de mascotas. Habla con empatía, usa un tono amigable pero profesional, y ocasionalmente puedes usar emojis de perros o gatos 🐶🐱 para dar calidez, pero sin perder la formalidad médica cuando sea necesario.
2. Reglas de Oro (Comportamiento General):
NUNCA des diagnósticos médicos, pronósticos ni recetas. Eres un asistente informativo, no un veterinario. Si el usuario describe síntomas 


**6. Dividir el texto en chunks**

In [32]:
#Importando desde LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [33]:
#CREANDO DOCUMENTOS DE LANGCHAIN
documentos_langchain = []
for documento in documentos_cargados:
    documentos_langchain.append(
        Document(
            page_content=documento["texto"],
            metadata=documento["metadata"]
        )
    )

print(
    f"Documentos convertidos al formato de LangChain: "
    f"{len(documentos_langchain)}"
)

Documentos convertidos al formato de LangChain: 6


In [34]:
#Configuración de divisor
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

In [35]:
#PARTE IMPORTANTE, CREANDO LOS CHUNKS
chunks = text_splitter.split_documents(documentos_langchain)

for indice, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = indice
    chunk.metadata["longitud_chunk"] = len(chunk.page_content)

print(f"Cantidad total de chunks: {len(chunks)}")

#Con seis documentos de ese tamaño, se podrá obtener aproximadamente entre 30 y 80 chunks.
#La cantidad exacta dependerá del contenido extraído, numero de caracteres, etc.

Cantidad total de chunks: 51


In [37]:
#INSPECCIÓN DE LOS PRIMEROS CHUNKS
for chunk in chunks[:5]:
    print("=" * 80)
    print("Chunk ID:", chunk.metadata["chunk_id"])
    print("Archivo:", chunk.metadata["nombre_archivo"])
    print("Categoría:", chunk.metadata["Categoria"])
    print("Longitud:", chunk.metadata["longitud_chunk"])
    print("\nContenido:")
    print(chunk.page_content)
    print()

Chunk ID: 0
Archivo: citas_agendamientos.docx
Categoría: Atención al cliente
Longitud: 933

Contenido:
PREGUNTAS FRECUENTES (FAQ) – CITAS Y AGENDAMIENTOS
VERSIÓN: 1.0
FECHA DE VIGENCIA: 8 de Octubre de 2026
EMPRESA: Veterinaria DOGtor S.A. de C.V.
DIRECCIÓN SEDE CENTRAL: Av. Paseo de la Reforma 1234, Colonia Juárez, Alcaldía Cuauhtémoc, CDMX.
Categoría: Atención al cliente
Departamento responsable: Recepción y Atención al Cliente
Tipo de documento: Preguntas Frecuentes
Nivel de acceso: Todos los colaboradores
Estado: Vigente
Palabras clave:
Citas, horarios, costos, agendar, urgencias
INTRODUCCIÓN
En Veterinaria DOGtor sabemos que la salud de tu peludo es prioridad. Por eso, hemos diseñado un sistema de agendamiento ágil, transparente y adaptado a tu ritmo de vida en la Ciudad de México. A continuación, resolvemos las dudas más comunes para que agendar, modificar o prepararte para tu cita sea pan comido (o mejor dicho, croquetas comidas).
1. ¿CÓMO PUEDO AGENDAR UNA CITA?
Contamos con 4 

In [38]:
#¿Cuanto chunks genera cada archivo?
#Con el siguiente codigo podemos saber
from collections import Counter

conteo_por_archivo = Counter(
    chunk.metadata["nombre_archivo"]
    for chunk in chunks
)

for archivo, cantidad in conteo_por_archivo.items():
    print(f"{archivo}: {cantidad} chunks")

citas_agendamientos.docx: 8 chunks
guia_convenios.docx: 10 chunks
instrucciones_pre_postconsulta.docx: 10 chunks
politica_cancelaciones_reprogramacion.docx: 9 chunks
politica_privacidad_datos_paciente.docx: 10 chunks
servicios_veterinaria_dogtor.docx: 4 chunks


In [39]:
#EL SIGUIENTE CODIGO ES PARA COMPROBAR SI NO EXISTEN CHUNKS VACIOS
chunks_vacios = [
    chunk for chunk in chunks
    if not chunk.page_content.strip()
]

print("Chunks vacíos:", len(chunks_vacios))

chunks_sin_archivo = [
    chunk for chunk in chunks
    if "nombre_archivo" not in chunk.metadata
]

print("Chunks sin nombre de archivo:", len(chunks_sin_archivo))

Chunks vacíos: 0
Chunks sin nombre de archivo: 0


In [40]:
#Las tablas pueden contener el error de no ser detectadas, por lo que, para saber si sobrevivieron al chunky
#se utiliza el siguiente codigo
chunks_con_tablas = [
    chunk for chunk in chunks
    if "[INICIO DE TABLA]" in chunk.page_content
    or "[FIN DE TABLA]" in chunk.page_content
]

print("Chunks relacionados con tablas:", len(chunks_con_tablas))

for chunk in chunks_con_tablas[:3]:
    print("=" * 80)
    print("Archivo:", chunk.metadata["nombre_archivo"])
    print(chunk.page_content)

Chunks relacionados con tablas: 9
Archivo: guia_convenios.docx
2. CONVENIOS CON ASEGURADORAS DE MASCOTAS (SEGUROS DE GASTOS MÉDICOS)
Aceptamos y facturamos directamente a las siguientes aseguradoras. Si tu póliza está activa, no pagas en caja (solo el deducible o coaseguro que marque tu contrato, si aplica).
[INICIO DE TABLA]
Aseguradora | Tipo de cobertura | Beneficio DOGtor | Proceso de cobro
GNP Seguros (Póliza Mascotas) | Accidentes, cirugías, hospitalización y estudios de diagnóstico. | Convenio preferente: Sin tope de gastos notariales para la reclamación. | Cobramos directamente a GNP. Tú solo firmas la factura y pagas el deducible establecido.
Mapfre México (Protección Animal) | Consultas de especialidad, medicamentos recetados y urgencias. | 10% de descuento en el deducible si pagas con tarjeta DOGtor (afiliación gratuita). | Emitimos factura CFDI a nombre de Mapfre y te damos el comprobante para tu ajuste.
Archivo: guia_convenios.docx
AXA Seguros (Salud para Mascotas) | Cober

**7. Generar embeddings**

In [41]:
#CREANDO EL CLIENTE DE GEMINI
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI")

if not GEMINI_API_KEY:
    raise ValueError(
        "No se encontró GEMINI_API_KEY en los secretos de Colab."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

print("Cliente de Gemini creado correctamente.")

Cliente de Gemini creado correctamente.


In [43]:
#Probando un Embedding
respuesta_prueba = client.models.embed_content(
    model="gemini-embedding-001",
    contents="Las cirugías deben cancelarse con anticipación."
)

vector_prueba = respuesta_prueba.embeddings[0].values

print("Embedding generado correctamente.")
print("Cantidad de dimensiones:", len(vector_prueba))
print("Primeros 10 valores:")
print(vector_prueba[:10])

Embedding generado correctamente.
Cantidad de dimensiones: 3072
Primeros 10 valores:
[-0.012572877, 0.015884738, -0.015732056, -0.05946917, -0.021752968, 0.0023923304, -0.021154229, 0.02685081, -0.00084704405, -0.005351702]


In [44]:
#Función para generar embeddings
def generar_embedding_documento(texto: str) -> list[float]:
    respuesta = client.models.embed_content(
        model="gemini-embedding-001",
        contents=texto
    )

    return respuesta.embeddings[0].values

**8. Guardarlos en Chroma**

In [46]:
#Creando una base persistente de Chroma
import chromadb
CHROMA_DIR = str(
    BASE_DIR / "chroma_db"
)

chroma_client = chromadb.PersistentClient(
    path=CHROMA_DIR
)

collection = chroma_client.get_or_create_collection(
    name="veterinaria_dogtor"
)

print("Colección creada:", collection.name)
print("Ruta:", CHROMA_DIR)
#El PersistentClient hará que la base no desaparezca al cerrar la sesión de Colab.

Colección creada: veterinaria_dogtor
Ruta: /content/drive/MyDrive/documentos_agente_de_ia/chroma_db


In [47]:
#Generando y almacenando los embeddings
import time

for indice, chunk in enumerate(chunks):
    chunk_id = f"chunk_{indice:04d}"

    embedding = generar_embedding_documento(
        chunk.page_content
    )

    collection.upsert(
        ids=[chunk_id],
        embeddings=[embedding],
        documents=[chunk.page_content],
        metadatas=[chunk.metadata]
    )

    print(
        f"[{indice + 1}/{len(chunks)}] "
        f"Guardado: {chunk_id} - "
        f"{chunk.metadata['nombre_archivo']}"
    )

    time.sleep(0.5)

[1/51] Guardado: chunk_0000 - citas_agendamientos.docx
[2/51] Guardado: chunk_0001 - citas_agendamientos.docx
[3/51] Guardado: chunk_0002 - citas_agendamientos.docx
[4/51] Guardado: chunk_0003 - citas_agendamientos.docx
[5/51] Guardado: chunk_0004 - citas_agendamientos.docx
[6/51] Guardado: chunk_0005 - citas_agendamientos.docx
[7/51] Guardado: chunk_0006 - citas_agendamientos.docx
[8/51] Guardado: chunk_0007 - citas_agendamientos.docx
[9/51] Guardado: chunk_0008 - guia_convenios.docx
[10/51] Guardado: chunk_0009 - guia_convenios.docx
[11/51] Guardado: chunk_0010 - guia_convenios.docx
[12/51] Guardado: chunk_0011 - guia_convenios.docx
[13/51] Guardado: chunk_0012 - guia_convenios.docx
[14/51] Guardado: chunk_0013 - guia_convenios.docx
[15/51] Guardado: chunk_0014 - guia_convenios.docx
[16/51] Guardado: chunk_0015 - guia_convenios.docx
[17/51] Guardado: chunk_0016 - guia_convenios.docx
[18/51] Guardado: chunk_0017 - guia_convenios.docx
[19/51] Guardado: chunk_0018 - instrucciones_pre_po

In [49]:
#Comprobando que todo haya quedado indexado
cantidad_indexada = collection.count()

print("Chunks generados:", len(chunks))
print("Registros en Chroma:", cantidad_indexada)

if cantidad_indexada == len(chunks):
    print("Todos los chunks fueron indexados correctamente.")
else:
    print(
        "La cantidad no coincide. "
        "Debemos revisar la indexación."
    )

Chunks generados: 51
Registros en Chroma: 51
Todos los chunks fueron indexados correctamente.


**9. Hacer búsquedas de prueba**

In [50]:
#Función para generar embeddings de preguntas
def generar_embedding_consulta(pregunta: str):
    respuesta = client.models.embed_content(
        model="gemini-embedding-001",
        contents=pregunta
    )
    return respuesta.embeddings[0].values

In [51]:
#La primera busqueda semántica
pregunta = "¿Puedo cancelar una cirugía un día antes?"

embedding_pregunta = generar_embedding_consulta(pregunta)

resultados = collection.query(
    query_embeddings=[embedding_pregunta],
    n_results=4
)

In [52]:
#RESULTADO del codigo anterior
print(resultados.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [55]:
#En este codigo, se buscará coincidencia con la pregunta: ¿Puedo cancelar una cirugía un día antes?, se mostrarán
#documentos que contengan información embedings sobre cancelar, cita, etc.
#En pocas palabras, busca ideas parecidas.
for i in range(len(resultados["documents"][0])):

    print("="*80)

    print(f"Resultado #{i+1}")

    print()

    print("Distancia:")
    print(resultados["distances"][0][i])

    print()

    print("Documento:")
    print(resultados["metadatas"][0][i]["nombre_archivo"])

    print()

    print(resultados["documents"][0][i][:900])

    print()

Resultado #1

Distancia:
0.5818626880645752

Documento:
politica_cancelaciones_reprogramacion.docx

Esta política aplica para todas las citas agendadas (consultas generales, especialidades, cirugías programadas, estudios de diagnóstico y consultas a domicilio), sin importar el canal por el que se hayan reservado (web, app, WhatsApp o telefónico).
2. PLAZOS Y COSTOS POR CANCELACIÓN
Dependiendo de con cuánta anticipación nos avises, aplican las siguientes reglas:
[INICIO DE TABLA]
Anticipación de la cancelación | ¿Qué sucede? | ¿Costo o penalización?
Más de 24 horas antes de la cita. | Cancelación gratuita. Liberamos el espacio para otro paciente. | Sin costo. El monto no cobrado queda a tu favor para futuras citas (si pagaste por adelantado, se convierte en saldo a favor).
Entre 12 y 24 horas antes de la cita. | Cancelación con aviso oportuno. | Sin penalización económica, pero te pedimos que agendes en un plazo máximo de 15 días naturales para no perder el beneficio de tu paquete o pro

In [58]:
#PROBANDO CON UNA SEGUNDA PREGUNTA
pregunta = "¿Hay algún convenio con aseguradoras?"

embedding_pregunta = generar_embedding_consulta(pregunta)

resultados = collection.query(
    query_embeddings=[embedding_pregunta],
    n_results=4
)
for i in range(len(resultados["documents"][0])):

    print("="*80)

    print(f"Resultado #{i+1}")

    print()

    print("Distancia:")
    print(resultados["distances"][0][i])

    print()

    print("Documento:")
    print(resultados["metadatas"][0][i]["nombre_archivo"])

    print()

    print(resultados["documents"][0][i][:900])


Resultado #1

Distancia:
0.477389395236969

Documento:
guia_convenios.docx

AXA Seguros (Salud para Mascotas) | Cobertura integral (incluye consultas generales y vacunación anual). | Cobertura ampliada: Incluye 2 consultas de seguimiento sin costo adicional (no cubiertas por la póliza base). | Tú pagas en mostrador y nosotros te damos el expediente clínico detallado para que lo presentes a AXA y te reembolsen en 48 hrs.
BBVA Seguros / Citi Banamex Afore (Pólizas de vida con extensión a mascotas) | Solo para emergencias graves (internamiento y cirugías mayores). | Gestión gratuita del papeleo ante la aseguradora. | Te acompañamos en todo el proceso de llenado de formatos para acelerar tu reembolso.
[FIN DE TABLA]
Requisito indispensable: Para usar tu seguro, debes presentar tu número de póliza y vigencia al momento del check-in en recepción. Si el siniestro ocurre en horario nocturno, nuestro médico de guardia se comunicará con la línea de autorización de la aseg
Resultado #2

Distancia

In [59]:
#Creando una función de búsqueda
#PRIMERA FUNCIÓN RAG
def buscar_contexto(
    pregunta: str,
    top_k: int = 4
):
    """
    Busca los fragmentos más relevantes
    para una pregunta.
    """

    embedding = generar_embedding_consulta(pregunta)

    resultados = collection.query(
        query_embeddings=[embedding],
        n_results=top_k
    )

    return resultados

In [63]:
#unción para mostrar resultados
def mostrar_resultados(resultados):

    documentos = resultados["documents"][0]
    metadatos = resultados["metadatas"][0]
    distancias = resultados["distances"][0]

    for i in range(len(documentos)):

        print("=" * 100)
        print(f"Resultado #{i+1}")

        print(f"\nDocumento : {metadatos[i]['nombre_archivo']}")
        print(f"Categoría : {metadatos[i]['Categoria']}")
        print(f"Distancia : {distancias[i]:.4f}")

        print("\nContenido:\n")

        print(documentos[i][:1000])

        print("\n")

In [64]:
#PREGUNTA SENCILLA
pregunta = "¿Cómo cancelo una cirugía?"
resultado = buscar_contexto(
    pregunta,
    top_k=4
)

mostrar_resultados(resultado)

Resultado #1

Documento : politica_cancelaciones_reprogramacion.docx
Categoría : Atención al Cliente
Distancia : 0.5745

Contenido:

4. PROCEDIMIENTO PARA CANCELAR O REPROGRAMAR
Para hacer efectiva tu solicitud, utiliza cualquiera de estos canales:
Autogestión (Recomendado): Ingresa a nuestro portal web o app, ve a "Mis citas" y selecciona la opción "Cancelar" o "Reprogramar". El sistema calculará automáticamente si aplica penalización.
WhatsApp Business: Envía un mensaje al +52 (55) 9876-5432 con tu número de cita (ej: CITA-1234) y la palabra "CANCELAR" o "REPROGRAMAR".
Vía telefónica: Llama al 55 1234-5678 en horario de oficina. Nuestro equipo procesa el cambio en el momento.
Para procedimientos quirúrgicos: La cancelación o reprogramación debe hacerse SÍ O SÍ por teléfono o presencialmente en recepción, ya que requiere la confirmación directa del médico cirujano.
5. EXCEPCIONES POR EMERGENCIAS REALES (FUERZA MAYOR)
Entendemos que en la CDMX pasan cosas. No aplicarán penalizaciones e

In [65]:
#PROBANDO CON PREGUNTAS MALICIOSAS
pregunta = "¿Qué hago antes de operar a mi perro?"
resultado = buscar_contexto(
    pregunta,
    top_k=4
)

mostrar_resultados(resultado)

Resultado #1

Documento : instrucciones_pre_postconsulta.docx
Categoría : Atención médica
Distancia : 0.4536

Contenido:

¿Qué pasa si mi mascota comió sin que yo supiera?
Comunícate de inmediato con nosotros. Si llegas y nos dices que comió, probablemente reprogramamos la cirugía por seguridad. Nunca arriesgamos la vida de tu peludo.
Checklist pre-quirúrgico (la noche anterior):
✅ Retirar el plato de comida a las 10:00 pm (si la cirugía es a las 10:00 am).
✅ Retirar el agua a las 6:00 am.
✅ Pasearlo bien antes de entrar para que vacíe vejiga e intestinos.
✅ Llevar su cobija o juguete favorito (con olor a casa) para que se sienta seguro al despertar.
4. INSTRUCCIONES PRE-ESTUDIOS DE LABORATORIO Y RADIOLOGÍA
Toma de muestras de sangre (química sanguínea, biometría): No es necesario ayuno para la mayoría de las pruebas, pero si es perfil hepático o lipídico, sí recomendamos 6 hrs de ayuno. Pregunta al agendar.
Radiografías (Rayos X): No requieren preparación especial, pero si es de tórax

**10. Conectar Gemini para redactar respuestas**

In [68]:
#CONTRUYENDO EL CONTEXTO
def construir_contexto(resultados):

    documentos = resultados["documents"][0]
    metadatos = resultados["metadatas"][0]

    contexto = []

    for i in range(len(documentos)):

        bloque = f"""
DOCUMENTO {i+1}

Archivo:
{metadatos[i]['nombre_archivo']}

Categoría:
{metadatos[i]['Categoria']}

Contenido:
{documentos[i]}
"""

        contexto.append(bloque)

    return "\n\n".join(contexto)

In [69]:
#Probando con una pregunta
pregunta = "¿Cómo cancelo una cirugía?"

resultado = buscar_contexto(pregunta)

contexto = construir_contexto(resultado)

print(contexto[:3000])


DOCUMENTO 1

Archivo:
politica_cancelaciones_reprogramacion.docx

Categoría:
Atención al Cliente

Contenido:
4. PROCEDIMIENTO PARA CANCELAR O REPROGRAMAR
Para hacer efectiva tu solicitud, utiliza cualquiera de estos canales:
Autogestión (Recomendado): Ingresa a nuestro portal web o app, ve a "Mis citas" y selecciona la opción "Cancelar" o "Reprogramar". El sistema calculará automáticamente si aplica penalización.
WhatsApp Business: Envía un mensaje al +52 (55) 9876-5432 con tu número de cita (ej: CITA-1234) y la palabra "CANCELAR" o "REPROGRAMAR".
Vía telefónica: Llama al 55 1234-5678 en horario de oficina. Nuestro equipo procesa el cambio en el momento.
Para procedimientos quirúrgicos: La cancelación o reprogramación debe hacerse SÍ O SÍ por teléfono o presencialmente en recepción, ya que requiere la confirmación directa del médico cirujano.
5. EXCEPCIONES POR EMERGENCIAS REALES (FUERZA MAYOR)
Entendemos que en la CDMX pasan cosas. No aplicarán penalizaciones en los siguientes casos,

In [70]:
#PASO IMPORTANTE: Construyendo el Prompt completo
def construir_prompt(
    pregunta,
    contexto
):

    prompt = f"""
{SYSTEM_PROMPT}

================================================

INFORMACIÓN RECUPERADA

================================================

{contexto}

================================================

PREGUNTA DEL USUARIO

================================================

{pregunta}

Responde únicamente utilizando la información proporcionada.
Si la respuesta no aparece en los documentos, indícalo claramente.
"""

    return prompt

In [71]:
#Probando el prompt
prompt = construir_prompt(
    pregunta,
    contexto
)

print(prompt[:5000])


PROMPT DE SISTEMA PARA EL AGENTE VETERINARIO DOGtor
Versión: 1.0 | Fecha: 26/07/2026
1. Rol y Personalidad:
Eres "Dokky", el asistente virtual oficial de Veterinaria DOGtor S.A. de C.V., ubicada en Av. Paseo de la Reforma 1234, CDMX. Tu misión es brindar información clara, cálida y precisa a los tutores (dueños) de mascotas. Habla con empatía, usa un tono amigable pero profesional, y ocasionalmente puedes usar emojis de perros o gatos 🐶🐱 para dar calidez, pero sin perder la formalidad médica cuando sea necesario.
2. Reglas de Oro (Comportamiento General):
NUNCA des diagnósticos médicos, pronósticos ni recetas. Eres un asistente informativo, no un veterinario. Si el usuario describe síntomas graves (vómito persistente, sangre, convulsiones), tu respuesta inmediata debe ser: "Esto parece una emergencia. Te recomiendo llamar de inmediato a nuestra línea de urgencias al 55 1234-5678 o acudir a nuestra sede en Reforma 1234. No esperes."
Mantén la confidencialidad. Nunca compartas datos de 

In [73]:
#CONECTANDO CON GEMINI

respuesta = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(respuesta.text)

Hola 🐶!

Para cancelar una cirugía, debes hacerlo **sí o sí por teléfono al 55 1234-5678 o presencialmente en nuestra recepción** de Reforma 1234. Esto es necesario porque requiere la confirmación directa del médico cirujano, según nuestra política de cancelaciones y reprogramación.

Es importante que consideres que las cirugías programadas tienen un plazo mínimo de 72 horas para cancelar sin penalización. Si se cancelan con menos de 72 horas, el costo de penalización será del 40% del valor total del procedimiento.

Si necesitas más ayuda, escríbenos al WhatsApp +52 (55) 9876-5432.


In [74]:
#ENCAPSULANDO TODO EN UN SOLO CODIGO
def responder(
    pregunta,
    top_k=4
):

    resultados = buscar_contexto(
        pregunta,
        top_k
    )

    contexto = construir_contexto(
        resultados
    )

    prompt = construir_prompt(
        pregunta,
        contexto
    )

    respuesta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return respuesta.text

In [75]:
respuesta = responder(
    "Hola Dokky, ¿Como estas?, sabes si van a operar a mi perrito?"
)

print(respuesta)

¡Hola! Qué gusto saludarte 🐶.

Como asistente virtual, no tengo acceso al historial médico individual de tu perrito ni a su agenda específica de procedimientos. Esa información es confidencial y solo la maneja directamente nuestro equipo veterinario y de recepción.

No tengo esa información a la mano, pero puedo comunicarte con nuestro equipo humano para que te ayuden a verificar si tu perrito tiene una cirugía programada. ¿Te parece si te paso a recepción? Puedes llamar directamente a nuestro equipo al 55 1234-5678 o escribirnos por WhatsApp al +52 (55) 9876-5432.


**10. Crear la interfaz**

In [76]:
#Instalacion de gradio
!pip install -qU gradio

In [77]:
#Verificando la instalación
import gradio as gr
print("Versión de Gradio:", gr.__version__)

Versión de Gradio: 6.20.0


In [79]:
#Creando una respuesta con fuentes
#Actualmente la función responder() devuelve solamente el texto generado por Gemini.
#por lo que es necesario crear otra función que también identifique los documentos recuperados:
def responder_con_fuentes(
    pregunta: str,
    top_k: int = 4
) -> str:
    """
    Ejecuta el pipeline RAG completo y devuelve
    la respuesta de Gemini junto con las fuentes consultadas.
    """

    if not pregunta or not pregunta.strip():
        return "Por favor, escribe una pregunta."

    # 1. Recuperación semántica
    resultados = buscar_contexto(
        pregunta=pregunta,
        top_k=top_k
    )

    # 2. Construcción del contexto
    contexto = construir_contexto(resultados)

    # 3. Construcción del prompt final
    prompt = construir_prompt(
        pregunta=pregunta,
        contexto=contexto
    )

    # 4. Generación de la respuesta
    respuesta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    texto_respuesta = respuesta.text

    # 5. Obtener los nombres de los documentos recuperados
    metadatos = resultados["metadatas"][0]

    fuentes = []

    for metadata in metadatos:
        nombre_archivo = metadata.get(
            "nombre_archivo",
            "Documento sin nombre"
        )

        if nombre_archivo not in fuentes:
            fuentes.append(nombre_archivo)

    # 6. Crear el bloque visual de fuentes
    fuentes_formateadas = "\n".join(
        f"- `{fuente}`"
        for fuente in fuentes
    )

    respuesta_final = f"""
{texto_respuesta}

---

### Fuentes consultadas

{fuentes_formateadas}
"""

    return respuesta_final

In [80]:
print(
    responder_con_fuentes(
        "¿Qué debo hacer para registrar a mi gato en la veterinaria?"
    )
)


¡Hola! Para registrar a tu gatito en la veterinaria y asegurar una primera consulta fluida, te pedimos traer algunos documentos y elementos esenciales. 🐱

Según nuestra sección de "PREGUNTAS FRECUENTES (FAQ) – CITAS Y AGENDAMIENTOS", para tu primera cita necesitarás:
*   Tu identificación oficial vigente (INE, Pasaporte o Cédula Profesional) y tu RFC para fines de facturación.
*   La Cartilla Nacional de Vacunación de tu mascota (si la tienes; si no, nosotros la generamos desde cero).
*   Si tu gatito ya tiene estudios previos (diagnósticos, rayos X, análisis de sangre de otra clínica), tráelos para que tengamos un historial completo.
*   Por seguridad de tu mascota y de los demás pacientes, por favor tráelo con una correa o en su transportadora.

¡Estamos emocionados de darle la bienvenida a tu gatito a la familia DOGtor! Si necesitas más ayuda o quieres agendar su primera cita, escríbenos al WhatsApp +52 (55) 9876-5432.

---

### Fuentes consultadas

- `citas_agendamientos.docx`
- `

In [86]:
#Adaptando la función para GRADIO
def chat_dokky(
    mensaje: str,
    historial: list
) -> str:
    """
    Función que conecta la interfaz de Gradio
    con el pipeline RAG de Dokky.
    """
#NOTA IMPORTANTE:
    try:
        return responder_con_fuentes(
            pregunta=mensaje,
            top_k=4
        )

    except Exception as error:
        print("Error interno:", repr(error))

        return (
            "Lo siento, ocurrió un error al procesar tu pregunta. "
            "Revisa la salida de la celda de Colab para obtener "
            "más detalles."
        )

In [88]:
#Creando la interfaz de Dokky
demo = gr.ChatInterface(
    fn=chat_dokky,

    title="🐶 Dokky AI",

    description=(
        "Hola, soy un Asistente virtual de la Veterinaria DOGtor. "
        "Consulta información sobre servicios, citas, convenios, "
        "cancelaciones e instrucciones pre y postconsulta."
    ),

    examples=[
        "¿Qué servicios ofrece la veterinaria?",
        "¿Cómo puedo reprogramar una cita?",
        "¿Qué debo hacer antes de una cirugía?",
        "¿Aceptan convenios o aseguradoras?",
        "¿Qué ocurre si cancelo una cita con poca anticipación?"
    ],

    textbox=gr.Textbox(
        placeholder="Escribe aquí tu pregunta sobre Veterinaria DOGtor...",
        container=False,
        scale=7
    ),

    chatbot=gr.Chatbot(
        height=500,
        placeholder=(
            "<h2>🐾 Hola, soy Dokky</h2>"
            "<p>Pregunta sobre los servicios y políticas "
            "de Veterinaria DOGtor.</p>"
            "<p>Cread@ por Andrés Contreras</p>"
        )
    )
)

In [ ]:
#LANZANDO LA INTERFAZ EN COLAB
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b621f5a8aedac27b42.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#Para detener la aplicación
demo.close()